# Entendimiento del Problema — Admisiones de Posgrado

**Autor:** Equipo SCO
**Fecha:** 2026-08-19

**Descripción:**
Análisis del Issue #1 "Descarga de los datos". Este notebook cubre el entendimiento
del problema, la carga reproducible del dataset RAW y una exploración inicial mínima.


## 🎯 Entendimiento del problema (respuestas requeridas por el Issue #1)

1. **Objetivo del problema:** predecir la probabilidad de admisión de un candidato a un programa
   de posgrado a partir de su perfil académico, para que los estudiantes estimen sus chances.
2. **Tipo de problema:** regresión supervisada (la variable objetivo es continua en el rango 0–1).
3. **Variable objetivo:** `Chance of Admit` (0 a 1).
4. **Significado de cada variable:**
   - `GRE Score` (0–340): puntaje del GRE.
   - `TOEFL Score` (0–120): puntaje del TOEFL.
   - `University Rating` (0–5): rating de la universidad.
   - `SOP` (0–5): fortaleza del Statement of Purpose.
   - `LOR` (0–5): fortaleza de la Letter of Recommendation.
   - `CGPA` (0–10): GPA de pregrado.
   - `Research` (0/1): experiencia de investigación.
   - `Chance of Admit` (0–1): probabilidad de admisión (target).
5. **Fuente del dataset:** `data/01_raw/Admission_Predict.csv` (623 filas). Corresponde al dataset
   clásico *Graduate Admissions*; la fuente exacta y la licencia no están documentadas en el repo.
6. **Estructura del repositorio:** template cookiecutter (`data/`, `models/`, `notebooks/1-data…8-reports`,
   `scripts/`, `src/`, `tests/`). Ver `AGENTS.md`.
7. **Requisitos del curso para el Issue #1:** obtención de datos RAW + entendimiento del problema.
   Resto de producción fuera de alcance.
8. **Contenido del notebook:** las respuestas de esta lista, lectura reproducible del RAW, exploración
   mínima y conclusión del tipo de problema.
9. **Ubicación del RAW:** `data/01_raw/` (inmutable).
10. **Criterios de aceptación:** el notebook ejecuta de punta a punta; responde las 14 preguntas;
    lee el RAW de forma reproducible sin modificarlo; PR con revisión y CI verde.
11. **Archivos afectados:** este notebook (modificado). Sin cambios en `src/`.
12. **Validaciones:** `ruff`, `pre-commit`, `pytest`, ejecución completa del notebook y revisión del `git diff`.
13. **Soluciones actuales (si las hay):** no documentadas en el repositorio; ni `README.md`,
    `AGENTS.md` ni `Informacion.txt` mencionan una solución existente para este problema.
14. **Métrica de desempeño (primera intuición):** al ser regresión con target continuo
    `Chance of Admit` (0–1), se propone el RMSE (raíz del error cuadrático medio) como métrica
    principal, complementada con MAE; ambas miden el error de predicción en la escala 0–1 del target.
15. **Alineación de la métrica con el objetivo:** el objetivo es estimar la probabilidad de admisión;
    RMSE/MAE cuantifican directamente qué tan lejos está la probabilidad predicha de la real, por lo
    que la métrica queda alineada con el objetivo.
16. **Desempeño mínimo necesario:** superar un baseline trivial (predecir siempre la media de
    `Chance of Admit`, ≈0.73). Ese baseline tendría un RMSE igual a la desviación estándar del target
    (≈0.14, según los datos); un modelo aporta valor si logra un RMSE menor.
17. **Problemas parecidos y reutilización:** el dataset corresponde al clásico *Graduate Admissions*
    (ver Referencias), ampliamente usado en ejercicios de regresión; es posible reutilizar enfoques y
    herramientas públicas aplicadas a ese dataset.
18. **Experiencia disponible:** no documentada en el repositorio.
19. **Resolución manual:** sin modelo, se podría estimar la admisión comparando el perfil del candidato
    (CGPA, GRE, TOEFL) contra promedios o umbrales históricos (p. ej. el perfil promedio de los admitidos),
    ordenando candidatos por CGPA/GRE y asignando una probabilidad aproximada según su posición relativa.
20. **Supuestos (hasta este momento):**
    - El dataset representa datos históricos de admisiones de posgrado.
    - La variable objetivo `Chance of Admit` es una probabilidad continua en [0, 1].
    - Las celdas vacías del CSV corresponden a valores faltantes (no a ceros) y se tratarán en etapas posteriores.
    - Los rangos de cada variable son los documentados en `Informacion.txt`.
    - El perfil académico (features) está disponible antes de la decisión de admisión; no hay fuga de
      información del target hacia las features.
    - El RAW (`data/01_raw/`) permanece inmutable.
21. **Cómo se actualizan los datos:** no documentado en el repositorio; el archivo
    `Admission_Predict.csv` es un snapshot estático sin proceso de actualización descrito.
22. **Frecuencia de actualización:** no documentada en el repositorio.


### Fuera de alcance en este Issue

No se realiza en el Issue #1: imputación de missing values, eliminación de outliers,
feature engineering, scaling, encoding, train/test split, baseline, entrenamiento de modelos,
model selection, MLflow, deployment, API, Docker ni pipelines FTI completos.


## 📚 Importar librerías


In [1]:
import sys

print(sys.executable)
print(sys.version)

/home/elkiruvi/Proyecto-Admisiones/.venv/bin/python3
3.12.13 (main, Aug  5 2026, 15:44:22) [Clang 22.1.3 ]


In [2]:
from pathlib import Path

import pandas as pd

## 💾 Carga reproducible del dataset RAW


In [3]:
def find_repo_root(start: Path) -> Path:
    """Localiza la raíz del repositorio subiendo hasta encontrar `pyproject.toml`."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del repositorio (pyproject.toml).")


ROOT = find_repo_root(Path.cwd())
DATA_PATH = ROOT / "data" / "01_raw" / "Admission_Predict.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset en {DATA_PATH}.")

# El RAW se lee sin modificar; pandas convierte las celdas vacías a NaN.
raw_df = pd.read_csv(DATA_PATH)

# 'LOR ' y 'Chance of Admit ' traen espacio final: se limpia SOLO en memoria.
raw_df.columns = raw_df.columns.str.strip()

print(f"Dataset cargado desde: {DATA_PATH}")
raw_df.head()

Dataset cargado desde: /home/elkiruvi/Proyecto-Admisiones/data/01_raw/Admission_Predict.csv


,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,337.0,118.0,4.0,4.5,4.5,9.65,1.0,0.92
1,324.0,107.0,4.0,4.0,4.5,8.87,1.0,0.76
2,316.0,104.0,3.0,3.0,3.5,8.00,1.0,0.72
3,322.0,110.0,3.0,3.5,2.5,8.67,1.0,0.80
4,314.0,103.0,2.0,2.0,3.0,8.21,0.0,0.65


## 🔍 Exploración inicial mínima


In [4]:
print(f"Shape (filas, columnas): {raw_df.shape}")
print(f"Columnas: {list(raw_df.columns)}")
print("\nTipos de datos:")
print(raw_df.dtypes)
print("\nValores faltantes por columna:")
print(raw_df.isna().sum())

Shape (filas, columnas): (623, 8)
Columnas: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit']

Tipos de datos:
GRE Score            float64
TOEFL Score          float64
University Rating    float64
SOP                  float64
LOR                  float64
CGPA                 float64
Research             float64
Chance of Admit      float64
dtype: object

Valores faltantes por columna:
GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
Chance of Admit       0
dtype: int64


## ✅ Conclusión sobre el tipo de problema

La variable objetivo `Chance of Admit` es continua (0–1), por lo que el problema es de **regresión**.
Los valores faltantes (7 de 8 columnas) se documentan aquí, pero se tratarán en etapas posteriores,
no en este Issue. El dataset RAW permanece inmutable.


## 📊 Conclusiones y próximos pasos

- El problema es de regresión; el target es `Chance of Admit`.
- El dataset RAW (623 filas) presenta valores faltantes en 7 de 8 columnas; se tratarán en etapas posteriores.
- Próximos pasos: exploración/EDA, tratamiento de missing values, feature engineering y baseline.


## 📖 Referencias

- Dataset *Graduate Admissions* (fuente original a confirmar): [Kaggle - Mohan S. Acharya](https://www.kaggle.com/datasets/mohansacharya/graduate-admissions)
